In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
import os
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        print(os.path.join(root, f))

In [5]:
!pip -q install GEOparse lifelines scikit-survival
import GEOparse, pandas as pd, numpy as np
import pandas as pd, glob
expr_g = pd.read_parquet(glob.glob('/kaggle/input/**/gse10846_expr.parquet', recursive=True)[0])
df = pd.read_csv(glob.glob('/kaggle/input/**/gse10846_clin.csv', recursive=True)[0], index_col=0)
print(expr_g.shape, df["event"].sum(), "events")  # expect (22880, 420) 165

(22880, 420) 165 events


In [6]:
!pip -q install GEOparse
import GEOparse
gse2 = GEOparse.get_GEO(geo="GSE117556", destdir="/kaggle/working")
g0 = list(gse2.gsms.values())[0]
g0.metadata["characteristics_ch1"]

30-Aug-2026 19:45:44 DEBUG utils - Directory /kaggle/working already exists. Skipping.
30-Aug-2026 19:45:44 INFO GEOparse - Downloading ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE117nnn/GSE117556/soft/GSE117556_family.soft.gz to /kaggle/working/GSE117556_family.soft.gz
100%|██████████| 343M/343M [00:01<00:00, 196MB/s]    
30-Aug-2026 19:45:45 DEBUG downloader - Size validation passed
30-Aug-2026 19:45:45 DEBUG downloader - Moving /tmp/tmph9_wzxx2 to /kaggle/working/GSE117556_family.soft.gz
30-Aug-2026 19:45:46 DEBUG downloader - Successfully downloaded ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE117nnn/GSE117556/soft/GSE117556_family.soft.gz
30-Aug-2026 19:45:46 INFO GEOparse - Parsing /kaggle/working/GSE117556_family.soft.gz: 
30-Aug-2026 19:45:46 DEBUG GEOparse - DATABASE: GeoMiame
30-Aug-2026 19:45:46 DEBUG GEOparse - SERIES: GSE117556
30-Aug-2026 19:45:46 DEBUG GEOparse - PLATFORM: GPL14951
30-Aug-2026 19:45:48 DEBUG GEOparse - SAMPLE: GSM3302930
30-Aug-2026 19:45:48 DEBUG GEOparse - SA

['clinic diagnosis: Diffuse Large B-cell Lymphoma',
 'molecular subtype: ABC',
 'molecular coo subtype: ABC']

In [7]:
# 1) scan ALL samples for any survival-looking keys
import collections
keys = collections.Counter()
for g in gse2.gsms.values():
    for c in g.metadata.get("characteristics_ch1", []):
        keys[c.split(":")[0].strip().lower()] += 1
print(keys)

# 2) check series-level supplementary files
print(gse2.metadata.get("supplementary_file", []))

Counter({'clinic diagnosis': 928, 'molecular subtype': 928, 'molecular coo subtype': 928})
['ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE117nnn/GSE117556/suppl/GSE117556_non-normlized.csv.gz']


In [ ]:
gse3 = GEOparse.get_GEO(geo="GSE31312", destdir="/kaggle/working")
g0 = list(gse3.gsms.values())[0]
print(g0.metadata["characteristics_ch1"])

In [9]:
import collections
keys = collections.Counter()
for g in gse3.gsms.values():
    for c in g.metadata.get("characteristics_ch1", []):
        keys[c.split(":")[0].strip().lower()] += 1
print(keys)
print(gse3.metadata.get("supplementary_file", []))

Counter({'gene expression profiling subgroup': 498, 'immunohistochemistry subgroup': 498})
['ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE31nnn/GSE31312/suppl/GSE31312_Microarray_and_clinical_data_DLBCL_475_cases_PMID_22437443.pdf.gz', 'ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE31nnn/GSE31312/suppl/GSE31312_RAW.tar']


In [11]:
import urllib.request

def peek_gsm_chars(gse_id, n_show=15):
    url = f"https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc={gse_id}&targ=gsm&form=text&view=brief"
    txt = urllib.request.urlopen(url).read().decode("utf-8", errors="ignore")
    chars = [l for l in txt.splitlines() if l.startswith("!Sample_characteristics_ch1")]
    print(f"{gse_id}: {len(chars)} characteristic lines")
    for l in chars[:n_show]:
        print("  ", l)
    return txt

txt4 = peek_gsm_chars("GSE181063")

GSE181063: 0 characteristic lines


In [13]:
import urllib.request, gzip, io

url = "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE181nnn/GSE181063/matrix/GSE181063_series_matrix.txt.gz"
resp = urllib.request.urlopen(url)
gz = gzip.GzipFile(fileobj=resp)

char_lines = []
for raw in gz:
    line = raw.decode("utf-8", errors="ignore")
    if line.startswith("!series_matrix_table_begin"):
        break
    if line.startswith("!Sample_characteristics_ch1"):
        char_lines.append(line)

print(f"{len(char_lines)} characteristics rows")
for l in char_lines:
    # show the field name from the first sample's entry
    first = l.split("\t")[1].strip().strip('"')
    print("  ", first[:80])

41 characteristics rows
   diagnostic_group: DLBCL
   case_diagnostic_subtype_icdo3: Diffuse large B-cell lymphoma, NOS
   qc_fail: 4
   probe_detection: 0.670196105
   reagent: MCS3
   microdissection: FALSE
   pred_combine: MHG
   conf_combine_mhg: 0.63
   conf_combine_abc: 0.00259
   conf_combine_gcb: 0.07955
   conf_combine_unc: 0.28786
   conf_entropy: 0.866345674
   age_at_diagnosis: 76.7
   Sex: F
   os_status: 1
   os_followup_y: 0.0985626283367556
   firstline_regimen: CHOP-R
   curative_intent: 1
   performance_status_ecog: 1
   b_symptoms: N
   Stage: IV
   ipi_score: 3
   ldh: raised
   albumin: 21
   hb: 10.8
   lymphs: 1.5
   wbc: 10.4
   num_extranodal: 1
   sample_pre_active_treatment: Yes
   in_pmid_30408148: FALSE
   in_pmid_32187361: FALSE
   
   
   
   
   
   
   
   
   
   


In [14]:
!wget -q -O /kaggle/working/gse181063_matrix.txt.gz \
  https://ftp.ncbi.nlm.nih.gov/geo/series/GSE181nnn/GSE181063/matrix/GSE181063_series_matrix.txt.gz

In [15]:
import urllib.request, gzip

# list the matrix dir to get the exact filename
idx = urllib.request.urlopen("https://ftp.ncbi.nlm.nih.gov/geo/series/GSE87nnn/GSE87371/matrix/").read().decode()
import re
fname = re.search(r'href="(GSE87371\S*series_matrix\S*\.txt\.gz)"', idx).group(1)
print(fname)

url = f"https://ftp.ncbi.nlm.nih.gov/geo/series/GSE87nnn/GSE87371/matrix/{fname}"
gz = gzip.GzipFile(fileobj=urllib.request.urlopen(url))
for raw in gz:
    line = raw.decode("utf-8", errors="ignore")
    if line.startswith("!series_matrix_table_begin"):
        break
    if line.startswith("!Sample_characteristics_ch1"):
        print("  ", line.split("\t")[1].strip().strip('"')[:80])

GSE87371_series_matrix.txt.gz
   ID: GHE0001A
   diagnosis: DLBCL
   age: 43
   Sex: MALE
   Stage: STAGE 1
   treatment: ACVBP
   ipi: 0
   age_adjusted_ipi: 0
   pfs_time: 41.3963039
   cens_pfs: 1
   os_time: 41.3963039
   cens_os: 1
   coo: GC


In [16]:
import gzip, pandas as pd, numpy as np

path = "/kaggle/working/gse181063_matrix.txt.gz"

# ---- pass 1: metadata rows ----
gsm_ids, char_rows = None, []
with gzip.open(path, "rt", errors="ignore") as f:
    for line in f:
        if line.startswith("!Sample_geo_accession"):
            gsm_ids = [x.strip().strip('"') for x in line.rstrip("\n").split("\t")[1:]]
        elif line.startswith("!Sample_characteristics_ch1"):
            char_rows.append([x.strip().strip('"') for x in line.rstrip("\n").split("\t")[1:]])
        elif line.startswith("!series_matrix_table_begin"):
            break

# each char row is "key: value" — build clinical frame
clin2 = pd.DataFrame(index=gsm_ids)
for row in char_rows:
    # key from first non-empty entry
    key = next(x for x in row if ":" in x).split(":",1)[0].strip().lower()
    vals = [x.split(":",1)[1].strip() if ":" in x else np.nan for x in row]
    if key in clin2.columns: key += "_2"
    clin2[key] = vals

print(clin2.shape)
print(clin2[["os_status","os_followup_y","firstline_regimen","ipi_score","qc_fail","diagnostic_group"]].head())

# ---- pass 2: expression block ----
expr2 = pd.read_csv(path, sep="\t", comment="!", index_col=0, compression="gzip")
expr2.columns = [c.strip('"') for c in expr2.columns]
print(expr2.shape)

(1310, 41)
           os_status       os_followup_y firstline_regimen ipi_score qc_fail  \
GSM5482779         1  0.0985626283367556            CHOP-R         3       4   
GSM5482780         1    10.2861054072553            CHOP-R         2       5   
GSM5482781         0    12.2819986310746            CHOP-R         3       0   
GSM5482782         1    5.18001368925393            CHOP-R         4       0   
GSM5482783         1    3.05270362765229             CVP-R         4       1   

           diagnostic_group  
GSM5482779            DLBCL  
GSM5482780            DLBCL  
GSM5482781            DLBCL  
GSM5482782            DLBCL  
GSM5482783            DLBCL  
(29372, 1310)


In [17]:
mask = (clin2["diagnostic_group"].str.contains("DLBCL", na=False)
        & clin2["qc_fail"].astype(str).isin(["0","0.0","FALSE"])
        & clin2["os_followup_y"].notna())
print(mask.sum(), "usable DLBCL samples")
print(clin2.loc[mask, "firstline_regimen"].value_counts())

991 usable DLBCL samples
firstline_regimen
CHOP-R                       709
Supportive/palliativeonly     94
CVP-R                         33
radiotherapyonly              24
CODOX-M(+/-R)                 22
CHOP                          21
CHOP-R/Bortezomib             21
Vincristine(+/-R)             15
CVP                            9
N                              8
GCVP                           6
Resectiononly(stageI)          6
DHAP-R                         3
IDARAM                         3
NEC                            3
COP                            2
AraC+Mx                        2
PmitCEBO/Rituximab             2
Chlorambucil(+/-R)             2
supportivecare                 1
Gemcitabine                    1
Etoposide                      1
COP/COPADM/CYM                 1
1                              1
DHAP/Rituximab                 1
Name: count, dtype: int64


In [18]:
# find the GPL id
with gzip.open(path, "rt", errors="ignore") as f:
    for line in f:
        if line.startswith("!Series_platform_id"):
            print(line); break

!Series_platform_id	"GPL14951"



In [19]:
expr2.to_parquet("/kaggle/working/gse181063_expr_probes.parquet")
clin2.to_csv("/kaggle/working/gse181063_clin.csv")

In [23]:
!wget -O /kaggle/working/GPL14951.soft.txt \
  "https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GPL14951&targ=self&form=text&view=full&mode=raw"

--2026-08-30 19:56:28--  https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GPL14951&targ=self&form=text&view=full&mode=raw
Resolving www.ncbi.nlm.nih.gov (www.ncbi.nlm.nih.gov)... 34.107.134.59, 2600:1901:0:c831::
Connecting to www.ncbi.nlm.nih.gov (www.ncbi.nlm.nih.gov)|34.107.134.59|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified [text/html]
Saving to: ‘/kaggle/working/GPL14951.soft.txt’

/kaggle/working/GPL     [ <=>                ]  20.97K  --.-KB/s    in 0s      

2026-08-30 19:56:29 (74.4 MB/s) - ‘/kaggle/working/GPL14951.soft.txt’ saved [21475]



In [24]:
import pandas as pd, io

with open("/kaggle/working/GPL14951.soft.txt", "rt", errors="ignore") as f:
    lines = f.readlines()
print(len(lines), "lines")

skip = next((i+1 for i, l in enumerate(lines) if l.startswith("!platform_table_begin")), None)
print("table starts at", skip)

ann = pd.read_csv(io.StringIO("".join(lines[skip:])), sep="\t", low_memory=False)
ann = ann[ann.iloc[:, 0].astype(str).str.startswith("ILMN")]
print(ann.shape)
print(ann.columns.tolist())

32 lines
table starts at None
(0, 1)
['<!doctype html><html lang="en-US" dir="ltr"><head><base href="https://www.google.com/recaptcha/challengepage/"><link rel="preconnect" href="//www.gstatic.com"><meta name="referrer" content="origin"><script nonce="R5fKmP5gGxcdpNbAqs-4iw">window[\'ppConfig\'] = {productName: \'RecaptchaChallengePageUi\', deleteIsEnforced:  true , sealIsEnforced:  true , heartbeatRate:  0.5 , periodicReportingRateMillis:  60000.0 , disableAllReporting:  false };(function(){\'use strict\';function k(a){var b=0;return function(){return b<a.length?{done:!1,value:a[b++]}:{done:!0}}}function l(a){var b=typeof Symbol!="undefined"&&Symbol.iterator&&a[Symbol.iterator];if(b)return b.call(a);if(typeof a.length=="number")return{next:k(a)};throw Error(String(a)+" is not an iterable or ArrayLike");}var m=typeof Object.defineProperties=="function"?Object.defineProperty:function(a,b,c){if(a==Array.prototype||a==Object.prototype)return a;a[b]=c.value;return a};']


In [25]:
!wget -q -O /kaggle/working/GPL14951_family.soft.gz \
  https://ftp.ncbi.nlm.nih.gov/geo/platforms/GPL14nnn/GPL14951/soft/GPL14951_family.soft.gz
import os; print(os.path.getsize("/kaggle/working/GPL14951_family.soft.gz")/1e6, "MB")

2423.190097 MB


In [26]:
import gzip, io, pandas as pd

buf, in_table = [], False
with gzip.open("/kaggle/working/GPL14951_family.soft.gz", "rt", errors="ignore") as f:
    for line in f:
        if line.startswith("!platform_table_begin"):
            in_table = True
            continue
        if line.startswith("!platform_table_end"):
            break
        if in_table:
            buf.append(line)

ann = pd.read_csv(io.StringIO("".join(buf)), sep="\t", low_memory=False)
print(ann.shape)
print(ann.columns.tolist())
ann.head()

(29377, 28)
['ID', 'Transcript', 'Species', 'Source', 'Search_Key', 'ILMN_Gene', 'Source_Reference_ID', 'RefSeq_ID', 'Entrez_Gene_ID', 'GI', 'Accession', 'Symbol', 'Protein_Product', 'Array_Address_Id', 'Probe_Type', 'Probe_Start', 'SEQUENCE', 'Chromosome', 'Probe_Chr_Orientation', 'Probe_Coordinates', 'Cytoband', 'Definition', 'Ontology_Component', 'Ontology_Process', 'Ontology_Function', 'Synonyms', 'Obsolete_Probe_Id', 'GB_ACC']


,ID,Transcript,Species,Source,Search_Key,ILMN_Gene,Source_Reference_ID,RefSeq_ID,Entrez_Gene_ID,GI,...,Probe_Chr_Orientation,Probe_Coordinates,Cytoband,Definition,Ontology_Component,Ontology_Process,Ontology_Function,Synonyms,Obsolete_Probe_Id,GB_ACC
0,ILMN_3166687,ILMN_333737,ILMN Controls,ILMN_Controls,ERCC-00162,ERCC-00162,ERCC-00162,NaN,NaN,NaN,...,NaN,NaN,NaN,Methanocaldococcus jannaschii spike-in control...,NaN,NaN,NaN,NaN,NaN,DQ516750
1,ILMN_3165566,ILMN_333646,ILMN Controls,ILMN_Controls,ERCC-00071,ERCC-00071,ERCC-00071,NaN,NaN,NaN,...,NaN,NaN,NaN,Synthetic construct clone NISTag13 external RN...,NaN,NaN,NaN,NaN,NaN,DQ883654
2,ILMN_3164811,ILMN_333584,ILMN Controls,ILMN_Controls,ERCC-00009,ERCC-00009,ERCC-00009,NaN,NaN,NaN,...,NaN,NaN,NaN,Synthetic construct clone TagJ microarray control,NaN,NaN,NaN,NaN,NaN,DQ668364
3,ILMN_3165363,ILMN_333628,ILMN Controls,ILMN_Controls,ERCC-00053,ERCC-00053,ERCC-00053,NaN,NaN,NaN,...,NaN,NaN,NaN,Methanocaldococcus jannaschii spike-in control...,NaN,NaN,NaN,NaN,NaN,DQ516785
4,ILMN_3166511,ILMN_333719,ILMN Controls,ILMN_Controls,ERCC-00144,ERCC-00144,ERCC-00144,NaN,NaN,NaN,...,NaN,NaN,NaN,Synthetic construct clone AG006.1100 external ...,NaN,NaN,NaN,NaN,NaN,DQ854995


In [27]:
probe2gene2 = ann.set_index("ID")["Symbol"].astype(str).str.strip()

expr2g = expr2.copy()
expr2g["gene"] = expr2g.index.map(probe2gene2)
expr2g = expr2g[expr2g["gene"].notna() & ~expr2g["gene"].isin(["nan", ""])]
v = expr2g.drop(columns="gene").var(axis=1)
expr2g = expr2g.loc[v.groupby(expr2g["gene"]).idxmax().values].set_index("gene")
print("GSE181063 genes:", expr2g.shape)

common_genes = expr_g.index.intersection(expr2g.index)
print(len(common_genes), "genes shared with GSE10846")

GSE181063 genes: (20817, 1310)
16011 genes shared with GSE10846


In [28]:
expr2g.to_parquet("/kaggle/working/gse181063_expr_genes.parquet")
clin2.to_csv("/kaggle/working/gse181063_clin.csv")
expr_g.to_parquet("/kaggle/working/gse10846_expr.parquet")
df.to_csv("/kaggle/working/gse10846_clin.csv")